In [4]:
import pandas as pd
import numpy as np

FILE_PATH = r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Raw Data\balancesheet.xlsx"

excel_file = pd.ExcelFile(FILE_PATH)

print("Sheets:")
print(excel_file.sheet_names)

Sheets:
['Balance Sheet']


In [5]:
raw = pd.read_excel(
    FILE_PATH,
    sheet_name="Balance Sheet",
    header=1
)

print("Shape:", raw.shape)

print("\nColumns:")
print(raw.columns.tolist())

display(raw.head())

Shape: (1312, 13)

Columns:
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']


,id,company_id,year,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
0,136,ABB,Dec 2012,21.0,626,0,260,907,109,1,0,798,907
1,137,ABB,Mar 2014,21.0,767,0,351,1139,98,1,0,1040,1139
2,138,ABB,Mar 2015,21.0,916,0,436,1374,96,4,0,1274,1374
3,139,ABB,Mar 2016,21.0,1174,0,421,1616,108,3,0,1505,1616
4,140,ABB,Mar 2017,21.0,1366,0,679,2066,110,6,0,1950,2066


In [6]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1312 entries, 0 to 1311
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1312 non-null   int64  
 1   company_id         1312 non-null   object 
 2   year               1312 non-null   object 
 3   equity_capital     1312 non-null   float64
 4   reserves           1312 non-null   int64  
 5   borrowings         1312 non-null   int64  
 6   other_liabilities  1312 non-null   int64  
 7   total_liabilities  1312 non-null   int64  
 8   fixed_assets       1312 non-null   int64  
 9   cwip               1312 non-null   int64  
 10  investments        1312 non-null   int64  
 11  other_asset        1312 non-null   int64  
 12  total_assets       1312 non-null   int64  
dtypes: float64(1), int64(10), object(2)
memory usage: 133.4+ KB


In [7]:
print("Duplicate IDs:", raw["id"].duplicated().sum())

print(
    "Duplicate company-year:",
    raw.duplicated(["company_id", "year"]).sum()
)

Duplicate IDs: 0
Duplicate company-year: 87


In [8]:
missing_report = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_pct": raw.isna().mean() * 100
})

display(
    missing_report.sort_values(
        "missing_count",
        ascending=False
    )
)

,missing_count,missing_pct
id,0,0.0
company_id,0,0.0
year,0,0.0
equity_capital,0,0.0
reserves,0,0.0
borrowings,0,0.0
other_liabilities,0,0.0
total_liabilities,0,0.0
fixed_assets,0,0.0
cwip,0,0.0


In [9]:
duplicates = raw[
    raw.duplicated(
        ["company_id", "year"],
        keep=False
    )
].sort_values(
    ["company_id", "year"]
)

print("Rows involved:", len(duplicates))
print("Duplicate company-year groups:",
      duplicates.groupby(
          ["company_id", "year"]
      ).ngroups)

display(duplicates.head(30))

Rows involved: 138
Duplicate company-year groups: 51


,id,company_id,year,equity_capital,reserves,borrowings,other_liabilities,total_liabilities,fixed_assets,cwip,investments,other_asset,total_assets
99,253,ASIANPAINT,Mar 2013,96.0,3288,251,3149,6784,2441,59,296,3989,6784
112,266,ASIANPAINT,Mar 2013,96.0,3288,251,3149,6784,2441,59,296,3989,6784
100,254,ASIANPAINT,Mar 2014,96.0,3943,249,3787,8075,2562,72,1424,4019,8075
113,267,ASIANPAINT,Mar 2014,96.0,3943,249,3787,8075,2562,72,1424,4019,8075
101,255,ASIANPAINT,Mar 2015,96.0,4646,418,3754,8914,2660,196,1588,4471,8914
114,268,ASIANPAINT,Mar 2015,96.0,4646,418,3754,8914,2660,196,1588,4471,8914
102,256,ASIANPAINT,Mar 2016,96.0,6429,323,3711,10559,3416,107,2712,4324,10559
115,269,ASIANPAINT,Mar 2016,96.0,6429,323,3711,10559,3416,107,2712,4324,10559
103,257,ASIANPAINT,Mar 2017,96.0,7508,560,4241,12405,3304,258,2652,6192,12405
116,270,ASIANPAINT,Mar 2017,96.0,7508,560,4241,12405,3304,258,2652,6192,12405


In [10]:
financial_columns = [
    "equity_capital",
    "reserves",
    "borrowings",
    "other_liabilities",
    "total_liabilities",
    "fixed_assets",
    "cwip",
    "investments",
    "other_asset",
    "total_assets"
]

duplicate_check = (
    duplicates
    .groupby(["company_id", "year"])[financial_columns]
    .nunique()
)

print("Groups with identical financial data:")
print((duplicate_check == 1).all(axis=1).sum())

print("Groups with different financial data:")
print((duplicate_check > 1).any(axis=1).sum())

Groups with identical financial data:
51
Groups with different financial data:
0


In [11]:
balance_clean = raw.drop_duplicates(
    subset=["company_id", "year"],
    keep="first"
).copy()

print("Rows before:", len(raw))
print("Rows after:", len(balance_clean))
print("Rows removed:", len(raw) - len(balance_clean))

Rows before: 1312
Rows after: 1225
Rows removed: 87


In [12]:
print(
    "Duplicate company-year after cleaning:",
    balance_clean.duplicated(
        ["company_id", "year"]
    ).sum()
)

print(
    "Duplicate IDs after cleaning:",
    balance_clean["id"].duplicated().sum()
)

Duplicate company-year after cleaning: 0
Duplicate IDs after cleaning: 0


In [13]:
balance_clean["balance_difference"] = (
    balance_clean["total_assets"]
    - balance_clean["total_liabilities"]
)

print(
    "Rows where Assets != Liabilities:",
    (balance_clean["balance_difference"] != 0).sum()
)

print(
    "Maximum absolute difference:",
    balance_clean["balance_difference"].abs().max()
)

Rows where Assets != Liabilities: 0
Maximum absolute difference: 0


In [14]:
balance_clean["liabilities_calculated"] = (
    balance_clean["equity_capital"]
    + balance_clean["reserves"]
    + balance_clean["borrowings"]
    + balance_clean["other_liabilities"]
)

balance_clean["liabilities_difference"] = (
    balance_clean["total_liabilities"]
    - balance_clean["liabilities_calculated"]
)

print(
    "Liability reconciliation failures:",
    (balance_clean["liabilities_difference"] != 0).sum()
)

Liability reconciliation failures: 443


In [15]:
balance_clean["assets_calculated"] = (
    balance_clean["fixed_assets"]
    + balance_clean["cwip"]
    + balance_clean["investments"]
    + balance_clean["other_asset"]
)

balance_clean["assets_difference"] = (
    balance_clean["total_assets"]
    - balance_clean["assets_calculated"]
)

print(
    "Asset reconciliation failures:",
    (balance_clean["assets_difference"] != 0).sum()
)

Asset reconciliation failures: 446


In [16]:
print(
    balance_clean[
        balance_clean["liabilities_difference"] != 0
    ][
        [
            "company_id",
            "year",
            "equity_capital",
            "reserves",
            "borrowings",
            "other_liabilities",
            "total_liabilities",
            "liabilities_calculated",
            "liabilities_difference"
        ]
    ].head(20)
)

    company_id      year  equity_capital  reserves  borrowings  \
2          ABB  Mar 2015            21.0       916           0   
6          ABB  Mar 2019            21.0      1987           0   
7          ABB  Mar 2020            21.0      2410         175   
8          ABB  Mar 2021            21.0      2581         153   
9          ABB  Mar 2022            21.0      2799         152   
10         ABB  Mar 2023            21.0      3167         113   
12         ABB  Sep 2024            21.0      3500          60   
13  ADANIENSOL  Mar 2014             0.1         0           0   
16  ADANIENSOL  Mar 2017          1100.0      1847        8975   
17  ADANIENSOL  Mar 2018          1100.0      4957       10428   
18  ADANIENSOL  Mar 2019          1100.0      6943       20137   
20  ADANIENSOL  Mar 2021          1100.0      7819       27095   
21  ADANIENSOL  Mar 2022          1100.0      8813       29902   
27    ADANIENT  Mar 2015           110.0     25618       83571   
29    ADAN

In [17]:
print(
    balance_clean[
        balance_clean["assets_difference"] != 0
    ][
        [
            "company_id",
            "year",
            "fixed_assets",
            "cwip",
            "investments",
            "other_asset",
            "total_assets",
            "assets_calculated",
            "assets_difference"
        ]
    ].head(20)
)

    company_id      year  fixed_assets   cwip  investments  other_asset  \
0          ABB  Dec 2012           109      1            0          798   
8          ABB  Mar 2021           251      1            0         3589   
15  ADANIENSOL  Mar 2016         10060    258           20         1407   
18  ADANIENSOL  Mar 2019         24412    694          336         7117   
21  ADANIENSOL  Mar 2022         30272   5060          561        11572   
22  ADANIENSOL  Mar 2023         32645   6200         1370        13716   
25    ADANIENT  Mar 2013         48770  29248          323        33529   
30    ADANIENT  Mar 2018         10555   5526         1461        38864   
32    ADANIENT  Mar 2020         10476   7347         1952        27099   
36    ADANIENT  Mar 2024         65978  35180         8701        50728   
37    ADANIENT  Sep 2024         84751  29745        10322        56949   
38  ADANIGREEN  Mar 2017          4341    267           26         1525   
40  ADANIGREEN  Mar 2019 

In [18]:
print(
    balance_clean["liabilities_difference"]
    .abs()
    .describe()
)

print(
    balance_clean["assets_difference"]
    .abs()
    .describe()
)

count    1225.000000
mean        1.912082
std        38.437312
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max      1201.000000
Name: liabilities_difference, dtype: float64
count    1225.000000
mean        0.530612
std         4.073979
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max       101.000000
Name: assets_difference, dtype: float64


In [19]:
# Remove temporary validation columns
balance_clean = balance_clean.drop(
    columns=[
        "liabilities_calculated",
        "liabilities_difference",
        "assets_calculated",
        "assets_difference",
        "balance_difference"
    ]
)

print("Final shape:", balance_clean.shape)
print("\nColumns:")
print(balance_clean.columns.tolist())

Final shape: (1225, 13)

Columns:
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']


In [20]:
balance_clean["debt_to_assets"] = np.where(
    balance_clean["total_assets"] != 0,
    balance_clean["borrowings"] / balance_clean["total_assets"],
    np.nan
)

balance_clean["debt_to_assets"] = (
    balance_clean["debt_to_assets"] * 100
)

In [21]:
balance_clean[
    [
        "company_id",
        "year",
        "borrowings",
        "total_assets",
        "debt_to_assets"
    ]
].head(10)

,company_id,year,borrowings,total_assets,debt_to_assets
0,ABB,Dec 2012,0,907,0.000000
1,ABB,Mar 2014,0,1139,0.000000
2,ABB,Mar 2015,0,1374,0.000000
3,ABB,Mar 2016,0,1616,0.000000
4,ABB,Mar 2017,0,2066,0.000000
5,ABB,Mar 2018,0,2416,0.000000
6,ABB,Mar 2019,0,2941,0.000000
7,ABB,Mar 2020,175,3547,4.933747
8,ABB,Mar 2021,153,3840,3.984375
9,ABB,Mar 2022,152,4224,3.598485


In [22]:
print(
    balance_clean["debt_to_assets"].describe()
)

print(
    "Negative debt-to-assets:",
    (balance_clean["debt_to_assets"] < 0).sum()
)

count    1224.000000
mean       29.691071
std        29.137127
min         0.000000
25%         2.654621
50%        22.700896
75%        47.734857
max        92.842686
Name: debt_to_assets, dtype: float64
Negative debt-to-assets: 0


In [23]:
from pathlib import Path

CLEANED_DATA = Path(
    r"C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data"
)

CLEANED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

output_file = CLEANED_DATA / "balance_sheet_clean.csv"

balance_clean.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: C:\Users\KIIT\OneDrive\Desktop\Financial Analytics Project\Cleaned Data\balance_sheet_clean.csv
